# 02 · Goal, plan, act

**The job:** when an agent gets it wrong, find out *where*: in its plan, in following its plan, or in wasted steps.
Without that, "the agent is wrong" is not something an engineer can fix.

This follows Snowflake's **Goal-Plan-Action (GPA)** way of breaking agent failures down. Every agent run has:

```
GOAL                 PLAN                       ACT
what was asked   →   the steps it decided   →   the tool calls it made, and the answer
```

And four things go wrong, each checked separately:

| Check | The question | Checked by |
| --- | --- | --- |
| Plan quality | Would this plan answer the question at all? | AI judge |
| Plan adherence | Did it do what it planned, and nothing else? | Code |
| Execution efficiency | Did it repeat calls, fail calls, burn tokens? | Code |
| Logical consistency | Does the answer follow from what the tools returned? | AI judge |

Rule of thumb: **use code where the rule is exact, a judge where it takes judgment.**

In [1]:
import json, logging
from google import genai
from google.genai import types

logging.getLogger("google_genai").setLevel(logging.ERROR)
client = genai.Client()                 # reads GOOGLE_API_KEY from your environment
JUDGE_MODEL = "gemini-flash-latest"

## The agent: a sales analyst

A manager asks: *"Which region grew fastest from Q2 to Q3 2026, and by how much?"*

The agent has a `query_sales(region, quarter)` tool, plus `web_search` and `make_chart` it can use if it wants.
Below are two recorded runs. **v1** is the current agent. **v2** is the same agent after one change to its planning
instruction: *"Before acting, list every number you need to answer."*

Each action records which plan step it served (`step`), whether it worked, and its tokens.

In [2]:
GOAL = "Which region grew fastest from Q2 to Q3 2026, and by how much?"

RUNS = {
    "v1": {
        "plan": ["Get sales for each region", "Answer with the top region"],
        "actions": [
            {"tool": "query_sales", "args": "EMEA, Q3", "result": "€4.2M", "ok": True, "step": 0, "tokens": 900},
            {"tool": "query_sales", "args": "EMEA, Q3", "result": "€4.2M", "ok": True, "step": 0, "tokens": 900},
            {"tool": "web_search", "args": "EMEA market growth 2026", "result": "article", "ok": True, "step": None, "tokens": 2400},
            {"tool": "query_sales", "args": "AMER, Q3", "result": "€5.1M", "ok": True, "step": 0, "tokens": 900},
            {"tool": "query_sales", "args": "APAC, Q3", "result": "timeout", "ok": False, "step": 0, "tokens": 300},
            {"tool": "make_chart", "args": "Q3 by region", "result": "chart.png", "ok": True, "step": None, "tokens": 1500},
        ],
        "answer": "AMER grew fastest, up 18% on last quarter.",
    },
    "v2": {
        "plan": ["Get Q2 sales for EMEA, AMER and APAC", "Get Q3 sales for EMEA, AMER and APAC",
                 "Compute growth per region", "Answer with the top region and its growth"],
        "actions": [
            {"tool": "query_sales", "args": "EMEA, Q2", "result": "€3.9M", "ok": True, "step": 0, "tokens": 900},
            {"tool": "query_sales", "args": "AMER, Q2", "result": "€4.6M", "ok": True, "step": 0, "tokens": 900},
            {"tool": "query_sales", "args": "APAC, Q2", "result": "€2.0M", "ok": True, "step": 0, "tokens": 900},
            {"tool": "query_sales", "args": "EMEA, Q3", "result": "€4.2M", "ok": True, "step": 1, "tokens": 900},
            {"tool": "query_sales", "args": "AMER, Q3", "result": "€5.1M", "ok": True, "step": 1, "tokens": 900},
            {"tool": "query_sales", "args": "APAC, Q3", "result": "€2.5M", "ok": True, "step": 1, "tokens": 900},
            {"tool": "calculate", "args": "growth per region", "result": "EMEA 7.7%, AMER 10.9%, APAC 25.0%", "ok": True, "step": 2, "tokens": 600},
        ],
        "answer": "APAC grew fastest: from €2.0M to €2.5M, up 25% quarter on quarter.",
    },
}

## Check 1 · Plan quality (AI judge)

Before looking at a single tool call: could this plan answer the question at all?

In [3]:
judge_tokens = 0

def ask_judge(prompt):
    global judge_tokens
    r = client.models.generate_content(
        model=JUDGE_MODEL, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0))
    judge_tokens += r.usage_metadata.total_token_count
    return json.loads(r.text)

def plan_quality(goal, plan):
    return ask_judge(f"""Goal: {goal}
Plan: {json.dumps(plan)}
Would carrying out this plan, step by step, produce a correct and complete answer to the goal?
Return JSON: {{"verdict": "GOOD" or "BAD", "reason": "<one short sentence>"}}""")

PLAN = {v: plan_quality(GOAL, run["plan"]) for v, run in RUNS.items()}
for v, res in PLAN.items():
    print(f"{v}: {res['verdict']}  {res['reason']}")

v1: BAD  The plan does not specify comparing sales between Q2 and Q3 2026 or calculating the growth amount.
v2: GOOD  The plan outlines all necessary steps to retrieve the relevant regional data, calculate growth rates, and identify the highest growth.


## Check 2 · Plan adherence (code)

Did it carry out every step it planned, and did it do things that were never in the plan?

In [4]:
def adherence(run):
    # a step counts as done only if every call made for it worked
    planned = [a for a in run["actions"] if a["step"] is not None]
    done = {a["step"] for a in planned} - {a["step"] for a in planned if not a["ok"]}
    # the last step is "answer", which is the final message, not a tool call
    tool_steps = range(len(run["plan"]) - 1)
    skipped = [run["plan"][i] for i in tool_steps if i not in done]
    off_plan = [a["tool"] for a in run["actions"] if a["step"] is None]
    return skipped, off_plan

for v, run in RUNS.items():
    skipped, off_plan = adherence(run)
    print(f"{v}: steps not completed {skipped or 'none'} · off-plan calls {off_plan or 'none'}")

v1: steps not completed ['Get sales for each region'] · off-plan calls ['web_search', 'make_chart']
v2: steps not completed none · off-plan calls none


## Check 3 · Execution efficiency (code)

Repeated calls, failed calls, and what the run cost. The price below is **illustrative**: use the current price of
your model (see [tokenomics-from-scratch](https://github.com/wilsoncelyCUC/tokenomics-from-scratch) for where to find it).

In [5]:
EUR_PER_MILLION_TOKENS = 2.00   # illustrative blended price, not a real quote

def efficiency(run):
    seen, repeats = set(), 0
    for a in run["actions"]:
        key = (a["tool"], a["args"])
        repeats += key in seen
        seen.add(key)
    failed = sum(not a["ok"] for a in run["actions"])
    tokens = sum(a["tokens"] for a in run["actions"])
    return repeats, failed, tokens, tokens * EUR_PER_MILLION_TOKENS / 1_000_000

for v, run in RUNS.items():
    repeats, failed, tokens, eur = efficiency(run)
    print(f"{v}: {len(run['actions'])} calls · {repeats} repeated · {failed} failed · {tokens:,} tokens · €{eur:.4f} per run")

v1: 6 calls · 1 repeated · 1 failed · 6,900 tokens · €0.0138 per run
v2: 7 calls · 0 repeated · 0 failed · 6,000 tokens · €0.0120 per run


v2 makes *more* calls but wastes none, and costs slightly less than v1.

Fewer calls is not the goal. **No wasted calls** is.

## Check 4 · Logical consistency (AI judge)

Is every claim in the answer supported by what the tools actually returned? This is where invented numbers get caught.

In [6]:
def consistency(goal, run):
    evidence = [f"{a['tool']}({a['args']}) -> {a['result']}" for a in run["actions"] if a["ok"]]
    return ask_judge(f"""Goal: {goal}
Tool results the agent received:
{chr(10).join(evidence)}
Agent's final answer: {run['answer']}
Is every claim and number in the answer directly supported by the tool results above?
Return JSON: {{"verdict": "CONSISTENT" or "INCONSISTENT", "unsupported": ["<claim>", ...], "reason": "<one short sentence>"}}""")

LOGIC = {v: consistency(GOAL, run) for v, run in RUNS.items()}
for v, res in LOGIC.items():
    print(f"{v}: {res['verdict']}  unsupported: {res['unsupported'] or 'none'}  ({res['reason']})")

v1: INCONSISTENT  unsupported: ['AMER grew fastest', 'up 18% on last quarter']  (The tool results do not contain Q2 sales figures or percentage growth calculations for any region.)
v2: CONSISTENT  unsupported: none  (All numbers, comparisons, and claims in the answer are directly corroborated by the tool results.)


## The scorecard

In [7]:
print(f"{'':4} {'plan quality':>13} {'adherence':>10} {'wasted calls':>13} {'consistent':>11} {'€ per run':>10}")
for v, run in RUNS.items():
    skipped, off_plan = adherence(run)
    repeats, failed, tokens, eur = efficiency(run)
    print(f"{v:4} {PLAN[v]['verdict']:>13} {'OK' if not (skipped or off_plan) else 'NO':>10}"
          f" {repeats + failed + len(off_plan):>13} {'YES' if LOGIC[v]['verdict'] == 'CONSISTENT' else 'NO':>11} {eur:>10.4f}")

print(f"\nRunning these two judge checks on 2 runs used {judge_tokens:,} tokens.")

      plan quality  adherence  wasted calls  consistent  € per run
v1             BAD         NO             4          NO     0.0138
v2            GOOD         OK             0         YES     0.0120

Running these two judge checks on 2 runs used 1,836 tokens.


## What this tells you

- **v1's answer was confidently wrong** ("AMER, up 18%"). It never fetched Q2 sales, so there was nothing to compute growth from.
  The root cause is the **plan**: "get sales for each region" never says *which quarters*.
  Everything after that (skipped data, off-plan searching, an invented percentage) follows from it.
- **One change to the planning instruction fixed all four checks.** That's why you check the plan first:
  it's usually the cheapest fix.
- **Evaluation costs tokens too.** Judge checks run on every test case, every release. Snowflake's own course warns that
  GPA checks "consume a significant number of tokens". Put the evaluation in the budget, and use code checks wherever a rule is exact.

## What you own as a program manager

1. **The goals.** The real questions your users ask, with the answer you'd accept.
2. **The order of investigation.** When a run fails: plan, then adherence, then efficiency, then the answer.
3. **The budget.** The cost per run *and* the cost of evaluating it.

## The same thing in TruLens

Snowflake implements these checks in the open-source [TruLens](https://github.com/truera/trulens) library:
it records each agent run as a trace and scores plan quality, plan adherence, execution efficiency and logical consistency
with judge models, in a dashboard you can compare across versions.

**Previous:** [01 · Judge the answer and the path](01-judge-the-answer-and-the-path.ipynb)